This file is used to generate the audio wav files. The audios contain multiple coughs per patient. ```extract_coughs``` extracts up to 5 1s coughs (224*224 samples) per patient. The coughs are saved in ```Extracted_cough_files``` using the patient's UID along with the cough number.

## Imports

In [1]:
import os
from scipy.io import wavfile
import numpy as np
import glob
import torch
import numpy as np
from scipy.fftpack import dct

In [2]:
def generate_spectrogram_mel_mfcc(audio, num_mel_bins=80, window_size=256, nfft=447, sample_rate=44100):  
    db_spectrogram = None
    db_mel_spectrogram = None
    mfcc = None
    
    if audio.shape[0] < sample_rate//2: audio = torch.nn.functional.pad(audio, (0, sample_rate//2-audio.shape[0]), mode="constant", value=audio.min())
    audio = audio[np.newaxis, :sample_rate//2]
    
    window = np.hanning(nfft)
    windowed_audio = [audio[0, win:win+nfft] * window for win in range(0, audio.shape[1]-nfft, window_size)]
    power_spec = (abs(np.fft.rfft(windowed_audio, n=nfft))**2) 
    power_spec_eps = np.where(power_spec == 0, 1e-9, power_spec) 
    db_spectrogram = 10*np.log10(power_spec_eps)    
    
    low_freq_mel = 0
    high_freq_mel = (2595 * np.log10(1 + (sample_rate / 2) / 700))  # Convert Hz to Mel
    mel_points = np.linspace(low_freq_mel, high_freq_mel, num_mel_bins + 2)  # Equally spaced in Mel scale
    hz_points = (700 * (10**(mel_points / 2595) - 1))  # Convert Mel to Hz
    bin = np.floor((nfft + 1) * hz_points / sample_rate)
    fbank = np.zeros((num_mel_bins, int(np.floor(nfft / 2 + 1))))
    for m in range(1, num_mel_bins + 1):
        f_m_minus = int(bin[m - 1])   # left
        f_m = int(bin[m])             # center
        f_m_plus = int(bin[m + 1])    # right

        for k in range(f_m_minus, f_m): fbank[m - 1, k] = (k - bin[m - 1]) / (bin[m] - bin[m - 1])
        for k in range(f_m, f_m_plus): fbank[m - 1, k] = (bin[m + 1] - k) / (bin[m + 1] - bin[m])
        
        if np.sum(fbank[m-1])==0: fbank[m-1] = fbank[m-2]/2+fbank[m]/2
    
    mel_spectrogram = np.dot(power_spec, fbank.T)
    mel_spectrogram = np.where(mel_spectrogram == 0, 1e-9, mel_spectrogram) 
    db_mel_spectrogram = 20*np.log10(mel_spectrogram)
    mfcc = dct(db_mel_spectrogram, type=2, axis=1, norm='ortho')
    
    return db_spectrogram, db_mel_spectrogram, mfcc

# HYFE DATA

In [3]:
'''cough_path = '../data/hyfe/audio'
num_mel_bins = 128
num_ceps = 39 
count = 0
for patient in glob.glob(os.path.join(cough_path, '*')):
    count += 1
    coughs = []
    mel_specs = []
    spec_mean, mel_spec_mean, mfcc_mean = 0,0,0
    
    for cough in glob.glob(patient+"/*.wav"):
        print(cough)
        UID = cough.rsplit("/")[5].replace("-recording-1.wav", "")
        patient_UID = cough.rsplit("/")[4]
        
        sample_rate, sample = wavfile.read(cough)
        audio = torch.tensor(sample)
        spec, mel_spec, mfcc = generate_spectrogram_mel_mfcc(audio, num_mel_bins=num_mel_bins, window_size=512, nfft=2048, sample_rate=44100)
        coughs.append([patient_UID, UID, spec, mel_spec, mfcc])

    for cough in coughs:
        patient_UID, UID, spec, mel_spec, mfcc = cough
        
        foldername_mel = "../data/hyfe/mel_spectrograms_"+str(num_mel_bins)+"/"+patient_UID
        filename = foldername_mel+"/"+UID
        if not os.path.exists(foldername_mel): os.makedirs(foldername_mel)        
        np.save(filename, mel_spec.T)'''

'cough_path = \'../data/hyfe/audio\'\nnum_mel_bins = 128\nnum_ceps = 39 \ncount = 0\nfor patient in glob.glob(os.path.join(cough_path, \'*\')):\n    count += 1\n    coughs = []\n    mel_specs = []\n    spec_mean, mel_spec_mean, mfcc_mean = 0,0,0\n\n    for cough in glob.glob(patient+"/*.wav"):\n        print(cough)\n        UID = cough.rsplit("/")[5].replace("-recording-1.wav", "")\n        patient_UID = cough.rsplit("/")[4]\n\n        sample_rate, sample = wavfile.read(cough)\n        audio = torch.tensor(sample)\n        spec, mel_spec, mfcc = generate_spectrogram_mel_mfcc(audio, num_mel_bins=num_mel_bins, window_size=512, nfft=2048, sample_rate=44100)\n        coughs.append([patient_UID, UID, spec, mel_spec, mfcc])\n\n    for cough in coughs:\n        patient_UID, UID, spec, mel_spec, mfcc = cough\n\n        foldername_mel = "../data/hyfe/mel_spectrograms_"+str(num_mel_bins)+"/"+patient_UID\n        filename = foldername_mel+"/"+UID\n        if not os.path.exists(foldername_mel): os

In [4]:
import os, glob, csv
import numpy as np

'''folder = "../data/hyfe/data_folds/"
if not os.path.exists(folder): os.makedirs(folder)

for fold in [1,2,3,4,5,6,7,8,9,10]:
    print(fold)
    f = open(folder+"/fold_"+str(fold-1)+".csv", "w")
    writer = csv.writer(f)
    writer.writerow(["Cough_ID", "Status"])

    data = []
    lines_meta = open("../data/hyfe/labels.csv", "r").readlines()[1:] # get label
    lines_list = open("../data/hyfe/lists/10_fold_cross_validation/"+str(fold)+".lst", "r").readlines() # get list
    
    for line_list in lines_list:
        patient_fold=line_list.strip()
        for line_meta in lines_meta:
            patient_meta, label = line_meta.rsplit(",")
            patient_meta = patient_meta.strip()

            if patient_fold==patient_meta:
                for patient in glob.glob("../data/hyfe/mel_spectrograms_128/*"):
                    if patient.rsplit("/")[-1].strip() == patient_fold:
                        patient_mel = patient.rsplit("/")[-1]
                        print(patient, patient_fold, patient_meta, patient_mel, label)
                        for cough in glob.glob(patient+"/*"):
                            cough_id = cough.rsplit("/")[-1].removesuffix(".npy")
                            data.append([patient_fold+"/"+cough_id.strip(), int(label)])
                            print([patient_fold+"/"+cough_id.strip(), int(label)])
        
    writer.writerows(data)
    f.close()
'''

'folder = "../data/hyfe/data_folds/"\nif not os.path.exists(folder): os.makedirs(folder)\n\nfor fold in [1,2,3,4,5,6,7,8,9,10]:\n    print(fold)\n    f = open(folder+"/fold_"+str(fold-1)+".csv", "w")\n    writer = csv.writer(f)\n    writer.writerow(["Cough_ID", "Status"])\n\n    data = []\n    lines_meta = open("../data/hyfe/labels.csv", "r").readlines()[1:] # get label\n    lines_list = open("../data/hyfe/lists/10_fold_cross_validation/"+str(fold)+".lst", "r").readlines() # get list\n\n    for line_list in lines_list:\n        patient_fold=line_list.strip()\n        for line_meta in lines_meta:\n            patient_meta, label = line_meta.rsplit(",")\n            patient_meta = patient_meta.strip()\n\n            if patient_fold==patient_meta:\n                for patient in glob.glob("../data/hyfe/mel_spectrograms_128/*"):\n                    if patient.rsplit("/")[-1].strip() == patient_fold:\n                        patient_mel = patient.rsplit("/")[-1]\n                        pr

# CAGE

In [5]:
cough_path = '../data/cage/audio'
speech_path = '../data/cage/counting'
num_ceps = 39

for num_mel_bins in [128]:
    # ------------------------------------------------------------
    # 1. Process COUGH audio files
    # ------------------------------------------------------------
    for patient in glob.glob(os.path.join(cough_path, '*')):
        print(f"Processing cough patient: {patient}")
        coughs = []
        mel_specs = []

        # Get list of .wav files
        wav_files = glob.glob(patient + "/*.wav")
        if not wav_files:
            print(f"  No .wav files found in {patient}, skipping...")
            continue

        for cough in wav_files:
            UID = cough.rsplit("/")[5].replace(".wav", "").replace("cough_", "")
            patient_UID = cough.rsplit("/")[4]

            sample_rate, sample = wavfile.read(cough)
            audio = torch.tensor(sample)
            spec, mel_spec, mfcc = generate_spectrogram_mel_mfcc(
                audio,
                num_mel_bins=num_mel_bins,
                window_size=512,
                nfft=2048,
                sample_rate=sample_rate   # use actual sample rate
            )
            coughs.append([patient_UID, UID, spec, mel_spec, mfcc])
            mel_specs.append(mel_spec)

        # Convert to array and compute stats (only if we have data)
        mel_specs = np.array(mel_specs)
        if mel_specs.size == 0:
            continue

        mel_mean = np.mean(mel_specs, axis=(0, 1))
        mel_std = np.std(mel_specs, axis=(0, 1))

        for count, cough in enumerate(coughs):
            patient_UID, UID, spec, mel_spec, mfcc = cough

            foldername_mel = f"../data/cage/mel_spectrograms_{num_mel_bins}/{patient_UID}"
            filename = f"{foldername_mel}/{UID}"
            if not os.path.exists(foldername_mel):
                os.makedirs(foldername_mel)
            np.save(filename, mel_spec.T)

    # ------------------------------------------------------------
    
    # 2. Process SPEECH (counting) audio files
    for patient in glob.glob(os.path.join(speech_path, '*')):
        print(f"Processing speech patient: {patient}")
        speech = []
        mel_specs = []

        wav_files = glob.glob(patient + "/*.wav")
        if not wav_files:
            print(f"  No .wav files found in {patient}, skipping...")
            continue

        for cough in wav_files:
            UID = cough.rsplit("/")[5].replace(".wav", "").replace("cough_", "")
            patient_UID = cough.rsplit("/")[4]

            sample_rate, sample = wavfile.read(cough)
            audio = torch.tensor(sample)
            spec, mel_spec, mfcc = generate_spectrogram_mel_mfcc(
                audio,
                num_mel_bins=num_mel_bins,
                window_size=512,
                nfft=2048,
                sample_rate=sample_rate   # use actual sample rate
            )
            speech.append([patient_UID, UID, spec, mel_spec, mfcc])
            mel_specs.append(mel_spec)

        mel_specs = np.array(mel_specs)
        if mel_specs.size == 0:
            continue

        mel_mean = np.mean(mel_specs, axis=(0, 1))
        mel_std = np.std(mel_specs, axis=(0, 1))

        for count, cough in enumerate(speech):
            patient_UID, UID, spec, mel_spec, mfcc = cough

            foldername_mel = f"../data/cage/mel_spectrograms_counting_{num_mel_bins}/{patient_UID}"
            filename = f"{foldername_mel}/{UID}"
            if not os.path.exists(foldername_mel):
                os.makedirs(foldername_mel)
            np.save(filename, mel_spec.T)

Processing cough patient: ../data/cage/audio/CAGE0048


/tmp/ipykernel_333967/2003204818.py:10: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  windowed_audio = [audio[0, win:win+nfft] * window for win in range(0, audio.shape[1]-nfft, window_size)]


Processing cough patient: ../data/cage/audio/CAGE0231
Processing cough patient: ../data/cage/audio/CAGE0487
Processing cough patient: ../data/cage/audio/CAGE0378
Processing cough patient: ../data/cage/audio/CAGE0403
Processing cough patient: ../data/cage/audio/CAGE0065
Processing cough patient: ../data/cage/audio/CAGE0315
Processing cough patient: ../data/cage/audio/CAGE0413
Processing cough patient: ../data/cage/audio/CAGE0007
Processing cough patient: ../data/cage/audio/CAGE0446
Processing cough patient: ../data/cage/audio/CAGE0455
Processing cough patient: ../data/cage/audio/CAGE0103
Processing cough patient: ../data/cage/audio/CAGE0017
Processing cough patient: ../data/cage/audio/CAGE0266
Processing cough patient: ../data/cage/audio/CAGE0043
Processing cough patient: ../data/cage/audio/CAGE0154
Processing cough patient: ../data/cage/audio/CAGE0429
Processing cough patient: ../data/cage/audio/CAGE0253
Processing cough patient: ../data/cage/audio/CAGE0381
Processing cough patient: ..

In [6]:
import os, glob, csv

folder = "../data/cage/data_folds_filtered"   # new folder to store filtered folds
if not os.path.exists(folder):
    os.makedirs(folder)

for fold in [0,1,2,3,4,5,6,7,8,9]:
    print(f"Processing fold {fold}")
    f = open(f"{folder}/fold_{fold}.csv", "w")
    writer = csv.writer(f)
    writer.writerow(["Cough_ID", "Status"])

    data = []
    # Read labels and fold list
    lines_meta = open("../data/cage/labels.csv", "r").readlines()[1:]
    lines = open(f"../data/cage/lists/10_fold_cross_validation/{fold+1}.lst", "r").readlines()

    for line_list in lines:
        patient_fold = line_list.strip()
        for line_meta in lines_meta:
            patient_meta, label = line_meta.rsplit(",", 1)
            patient_meta = patient_meta.strip()
            if patient_fold == patient_meta:
                # --- Check for both cough and speech audio ---
                cough_dir = f"../data/cage/audio/{patient_fold}"
                speech_dir = f"../data/cage/counting/{patient_fold}"
                has_cough = len(glob.glob(os.path.join(cough_dir, "*.wav"))) > 0
                has_speech = len(glob.glob(os.path.join(speech_dir, "*.wav"))) > 0
                if not (has_cough and has_speech):
                    # Skip this patient – missing one modality
                    continue

                # If both exist, add all cough files
                for cough_file in glob.glob(os.path.join(cough_dir, "*.wav")):
                    cough_id = cough_file.rsplit("/", 1)[-1].removesuffix(".wav").removeprefix("cough_")
                    data.append([f"{patient_fold}/{cough_id}", int(label)])

    writer.writerows(data)
    f.close()

Processing fold 0
Processing fold 1
Processing fold 2
Processing fold 3
Processing fold 4
Processing fold 5
Processing fold 6
Processing fold 7
Processing fold 8
Processing fold 9
